# 23.8 — Regret minimization & CFR

Regret minimization learns equilibrium behavior by repeatedly asking which action would have done better in hindsight. Save a copy to Drive to edit.

The notebook builds external regret, regret matching, and a small counterfactual-regret minimization loop. The D5 rung emphasizes the lesson's pitfall: local CFR regrets need reach weights, and current policies can cycle while average strategies stabilize.

In [ ]:
import itertools
import math

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(23)

np.set_printoptions(precision=3, suppress=True)


## The concept, built once: external regret and regret matching

External regret is $R_T=\max_a\sum_t u_t(a)-\sum_t u_t(a_t)$. For the lesson's four rounds with payoffs $[1,-1],[-1,1],[1,-1],[-1,1]$ and actions H,H,T,T, the algorithm payoff and both fixed-action payoffs are $0$, so regret is exactly $0$.

In [ ]:
def external_regret(payoffs, actions):
    payoffs = np.asarray(payoffs, dtype=float)
    actions = np.asarray(actions, dtype=int)
    actual = float(np.sum(payoffs[np.arange(len(actions)), actions]))
    fixed = payoffs.sum(axis=0)
    regret = float(np.max(fixed) - actual)
    return regret, actual, fixed


def regret_matching(regrets):
    regrets = np.asarray(regrets, dtype=float)
    positive = np.maximum(regrets, 0.0)
    total = float(np.sum(positive))
    if total <= 0.0:
        return np.full_like(positive, 1.0 / len(positive), dtype=float)
    return positive / total


lesson_payoffs = np.array([
    [1.0, -1.0],
    [-1.0, 1.0],
    [1.0, -1.0],
    [-1.0, 1.0],
])
lesson_actions = np.array([0, 0, 1, 1])
lesson_regret, lesson_actual, lesson_fixed = external_regret(lesson_payoffs, lesson_actions)
lesson_policy = regret_matching(np.array([3.0, 1.0]))

assert round(lesson_actual, 6) == 0.0
assert np.allclose(lesson_fixed, np.array([0.0, 0.0]))
assert round(lesson_regret, 6) == 0.0
assert np.allclose(lesson_policy, np.array([0.75, 0.25]))

print("External regret:", lesson_regret)
print("Regret-matching policy:", lesson_policy)


## One information set: counterfactual regret

At an information set with reach probability $0.25$, action values $(2,-1)$, and local strategy $(0.5,0.5)$, the node value is $0.5$. Counterfactual regrets are $0.25(2-0.5)=0.375$ and $0.25(-1-0.5)=-0.375$.

In [ ]:
def counterfactual_regret(reach_probability, action_values, strategy):
    action_values = np.asarray(action_values, dtype=float)
    strategy = np.asarray(strategy, dtype=float)
    node_value = float(strategy @ action_values)
    regrets = reach_probability * (action_values - node_value)
    return regrets, node_value


cf_regrets, cf_node_value = counterfactual_regret(0.25, np.array([2.0, -1.0]), np.array([0.5, 0.5]))

assert round(cf_node_value, 6) == 0.5
assert np.allclose(cf_regrets, np.array([0.375, -0.375]))

print("Node value:", cf_node_value)
print("Counterfactual regrets:", cf_regrets)


## Dataset ladder: D1–D5 regret problems of rising complexity

D1 is the lesson's four-round table. D2 is repeated matching pennies, D3 is an adversarial sequence that cycles current policies, D4 has multiple information sets, and D5 runs a compact Kuhn-poker CFR implementation with reach-weighted regrets and average strategies.

In [ ]:
def matching_pennies_sequence(rounds):
    payoffs = np.zeros((rounds, 2))
    for t in range(rounds):
        if t % 2 == 0:
            payoffs[t] = np.array([1.0, -1.0])
        else:
            payoffs[t] = np.array([-1.0, 1.0])
    return payoffs


def build_regret_ladder():
    d1 = {
        "name": "D1 four-round lesson table",
        "payoffs": lesson_payoffs,
        "actions": lesson_actions,
        "kind": "external",
    }
    d2_payoffs = matching_pennies_sequence(40)
    d2 = {
        "name": "D2 repeated matching pennies",
        "payoffs": d2_payoffs,
        "actions": np.arange(40) % 2,
        "kind": "external",
    }
    d3_payoffs = np.vstack([np.tile([1.0, -1.0], (25, 1)), np.tile([-1.0, 1.0], (25, 1))])
    d3 = {
        "name": "D3 adversarial cycling sequence",
        "payoffs": d3_payoffs,
        "actions": np.concatenate([np.zeros(25, dtype=int), np.ones(25, dtype=int)]),
        "kind": "external",
    }
    d4 = {
        "name": "D4 two information sets",
        "infosets": [
            {"reach": 0.4, "values": np.array([1.2, -0.2]), "strategy": np.array([0.6, 0.4])},
            {"reach": 0.15, "values": np.array([-0.5, 1.5]), "strategy": np.array([0.5, 0.5])},
        ],
        "kind": "cfr_local",
    }
    d5 = {
        "name": "D5 reach-weighted Kuhn CFR",
        "iterations": 300,
        "kind": "kuhn_cfr",
    }
    return [d1, d2, d3, d4, d5]


regret_ladder = build_regret_ladder()
for rung in regret_ladder:
    print(rung["name"])
    if rung["kind"] == "external":
        print("  rounds/actions:", rung["payoffs"].shape, len(rung["actions"]))
    elif rung["kind"] == "cfr_local":
        print("  infosets:", len(rung["infosets"]))
    else:
        print("  iterations:", rung["iterations"])


In [ ]:
class KuhnNode:
    def __init__(self, key):
        self.key = key
        self.regret_sum = np.zeros(2)
        self.strategy_sum = np.zeros(2)

    def strategy(self, reach_weight):
        strategy = regret_matching(self.regret_sum)
        self.strategy_sum += reach_weight * strategy
        return strategy

    def average_strategy(self):
        total = np.sum(self.strategy_sum)
        if total <= 0.0:
            return np.array([0.5, 0.5])
        return self.strategy_sum / total


def kuhn_terminal_utility(history, cards, player):
    opponent = 1 - player
    if history.endswith("pp"):
        return 1.0 if cards[player] > cards[opponent] else -1.0
    if history.endswith("bp"):
        bettor = 0 if history == "bp" else 1
        return 1.0 if player == bettor else -1.0
    if history.endswith("bb"):
        return 2.0 if cards[player] > cards[opponent] else -2.0
    return None


def kuhn_cfr(cards, history, reach0, reach1, nodes, player):
    terminal = kuhn_terminal_utility(history, cards, player)
    if terminal is not None:
        return terminal

    current_player = len(history) % 2
    key = str(cards[current_player]) + history
    if key not in nodes:
        nodes[key] = KuhnNode(key)
    node = nodes[key]
    reach_weight = reach0 if current_player == 0 else reach1
    strategy = node.strategy(reach_weight)
    utilities = np.zeros(2)
    actions = ["p", "b"]

    for action_index, action in enumerate(actions):
        next_history = history + action
        if current_player == 0:
            utilities[action_index] = -kuhn_cfr(cards, next_history, reach0 * strategy[action_index], reach1, nodes, 1 - player)
        else:
            utilities[action_index] = -kuhn_cfr(cards, next_history, reach0, reach1 * strategy[action_index], nodes, 1 - player)

    node_value = float(strategy @ utilities)
    regrets = utilities - node_value
    if current_player == 0:
        node.regret_sum += reach1 * regrets
    else:
        node.regret_sum += reach0 * regrets

    return node_value


def run_kuhn_cfr(iterations, seed=238):
    local_rng = np.random.default_rng(seed)
    nodes = {}
    cards_all = np.array([0, 1, 2])
    utilities = []
    exploit_proxy = []
    for t in range(iterations):
        cards = local_rng.permutation(cards_all)[:2]
        utility = kuhn_cfr(cards, "", 1.0, 1.0, nodes, 0)
        utilities.append(utility)
        average_root = []
        for card in cards_all:
            key = str(card)
            if key in nodes:
                average_root.append(nodes[key].average_strategy()[1])
        if average_root:
            exploit_proxy.append(float(np.std(average_root)))
        else:
            exploit_proxy.append(0.0)
    return nodes, np.array(utilities), np.array(exploit_proxy)


regret_results = []
for rung in regret_ladder:
    if rung["kind"] == "external":
        regret, actual, fixed = external_regret(rung["payoffs"], rung["actions"])
        metric = regret / len(rung["actions"])
        regret_results.append({"name": rung["name"], "size": len(rung["actions"]), "metric": metric, "detail": regret})
    elif rung["kind"] == "cfr_local":
        local_regrets = []
        for infoset in rung["infosets"]:
            regrets, node_value = counterfactual_regret(infoset["reach"], infoset["values"], infoset["strategy"])
            local_regrets.append(np.max(np.maximum(regrets, 0.0)))
        metric = float(np.mean(local_regrets))
        regret_results.append({"name": rung["name"], "size": len(rung["infosets"]), "metric": metric, "detail": sum(local_regrets)})
    else:
        nodes, utilities, exploit_proxy = run_kuhn_cfr(rung["iterations"])
        metric = float(np.mean(exploit_proxy[-50:]))
        regret_results.append({"name": rung["name"], "size": rung["iterations"], "metric": metric, "detail": len(nodes), "nodes": nodes, "exploit": exploit_proxy})

print("rung | size | average regret / exploitability proxy | detail")
for result in regret_results:
    print(result["name"], result["size"], round(result["metric"], 4), result["detail"])


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for col, rung in enumerate(regret_ladder):
    ax = axes[col]
    if rung["kind"] == "external":
        cumulative = np.cumsum(rung["payoffs"], axis=0)
        ax.plot(cumulative[:, 0], label="H")
        ax.plot(cumulative[:, 1], label="T")
        ax.set_title(rung["name"].split()[0] + " payoff")
        ax.legend(fontsize=7)
    elif rung["kind"] == "cfr_local":
        values = []
        for infoset in rung["infosets"]:
            regrets, node_value = counterfactual_regret(infoset["reach"], infoset["values"], infoset["strategy"])
            values.append(regrets)
        matrix = np.vstack(values)
        image = ax.imshow(matrix, cmap="coolwarm", vmin=-0.5, vmax=0.5)
        ax.set_title("D4 local regrets")
    else:
        exploit = regret_results[col]["exploit"]
        ax.plot(exploit)
        ax.set_title("D5 CFR proxy")
    ax.set_xlabel("round / action")

fig.tight_layout()
plt.show()

sizes = [result["size"] for result in regret_results]
metrics = [result["metric"] for result in regret_results]

plt.figure(figsize=(7, 4))
plt.plot(sizes, metrics, marker="o")
plt.xscale("log")
plt.xlabel("rounds / tree size")
plt.ylabel("average regret or exploitability proxy")
plt.title("Regret metric vs. problem size")
plt.grid(True, alpha=0.3)
plt.show()


## Pitfall on D5: dropping reach weights and average strategies

The wrong local update uses raw action-value differences no matter how often an information set is reached. The fix multiplies regret by the opponent reach probability and reports average strategies, because current regret-matching policies may keep cycling.

In [ ]:
reach = 0.25
values = np.array([2.0, -1.0])
strategy = np.array([0.5, 0.5])
wrong_regrets = values - strategy @ values
fixed_regrets, fixed_node_value = counterfactual_regret(reach, values, strategy)
d5_nodes = regret_results[-1]["nodes"]
average_strategies = {key: node.average_strategy() for key, node in sorted(d5_nodes.items())[:6]}
current_strategies = {key: regret_matching(node.regret_sum) for key, node in sorted(d5_nodes.items())[:6]}

print("Wrong unweighted local regrets:", wrong_regrets)
print("Fixed reach-weighted local regrets:", fixed_regrets)
print("Average strategies, first nodes:", average_strategies)
print("Current strategies, first nodes:", current_strategies)
print("Fix: use reach weights for local regrets and average strategies for reporting.")


## Evaluate it + Practice

- Metric: average external regret for online sequences and an exploitability proxy for CFR; compare against a no-learning uniform policy.
- Cheap sanity check: regret-matching probabilities must sum to one, and zero positive regrets must return uniform probabilities.
- Ablation: remove reach weights in D5 and inspect how rare information sets dominate the update.
- Failure signals: last-round chasing, reporting only current policy, or using local regrets without counterfactual reach.

Practice prompt 1: Change the D2 action sequence and compute average external regret.

Practice prompt 2: Run regret matching on cumulative regrets with one negative and one positive action.

Practice prompt 3: Increase D5 iterations and compare current versus average root strategies.